In [ ]:
# -----------------------------
# 1. ENVIRONMENT SETUP
# -----------------------------
import sys
import os

sys.path.append(os.path.abspath('../util'))

from FM_analyzer import FMAnalyzer

analyzer = FMAnalyzer(
    matrix_npz_path="./output/connectome_matrix.npz",
    neuron_ids_path="./output/neuron_ids.npy",
    neuron_types_path="../data/visual_neuron_types.txt"
)

analyzer.export_combined_summary_csv(
    side="right",
    neuron_types=["L1","L2","L3"],
    pos_npz_paths=[
        "./output/l1_right_excitatory.npz",
        "./output/l2_right_excitatory.npz",
        "./output/l3_right_excitatory.npz"
    ],
    neg_npz_paths=[
        "./output/l1_right_inhibitory.npz",
        "./output/l2_right_inhibitory.npz",
        "./output/l3_right_inhibitory.npz"
    ],
    cell_type_path="../data/cell_type.txt",
    output_csv="./output/visual_right_weight_summary.csv"
)

analyzer.export_combined_summary_csv(
    side="left",
    neuron_types=["L1","L2","L3"],
    pos_npz_paths=[
        "./output/l1_left_excitatory.npz",
        "./output/l2_left_excitatory.npz",
        "./output/l3_left_excitatory.npz"
    ],
    neg_npz_paths=[
        "./output/l1_left_inhibitory.npz",
        "./output/l2_left_inhibitory.npz",
        "./output/l3_left_inhibitory.npz"
    ],
    cell_type_path="../data/cell_type.txt",
    output_csv="./output/visual_left_weight_summary.csv"
)

# Fig. 2(a,b,c,d)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import scienceplots
from matplotlib.ticker import ScalarFormatter

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 28,
    "axes.labelsize": 28,
    "axes.titlesize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "axes.linewidth": 1.5,
})

def exp_func(x, a, b):
    return a * np.exp(-b * x)

def plot_Strengths_4_figs(left_csv, right_csv, save_path_prefix=None):
    df_left = pd.read_csv(left_csv)
    df_right = pd.read_csv(right_csv)
    df_left = df_left[df_left["min_depth"] >= 0]
    df_right = df_right[df_right["min_depth"] >= 0]

    depths_left = np.sort(df_left["min_depth"].unique())
    depths_right = np.sort(df_right["min_depth"].unique())

    # ---------------- a ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        counts = df["min_depth"].value_counts().sort_index()
        ax.bar(counts.index, counts.values, color="tab:gray", alpha=0.7, width=0.6)
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel("Count")
        ax.set_xticks(counts.index)
        ax.set_xticklabels(counts.index, rotation=0)  
        ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
        ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_count.png", dpi=300)

    # ---------------- b ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        data_by_depth = [df[df["min_depth"]==d]["max_FM_weight"].values for d in depths]
        means = np.array([np.mean(v) for v in data_by_depth])
        
        ax.boxplot(
            data_by_depth, 
            positions=depths, 
            widths=0.6, 
            patch_artist=True,
            boxprops=dict(facecolor="tab:blue", alpha=0.6), 
            showfliers=False,
            medianprops=dict(color='black', linewidth=1.5),
            whiskerprops=dict(color='tab:blue', linestyle='--'),
            capprops=dict(color='tab:blue')
        )
        
        ax.plot(depths, means, color='black', linewidth=2, label='mean')
        
        try:
            popt, _ = curve_fit(exp_func, depths, means, p0=(max(means),0.1))
            x_fit = np.linspace(min(depths), max(depths), 200)
            y_fit = exp_func(x_fit, *popt)
            ax.plot(x_fit, y_fit, 'r--', linewidth=2, label='exp fit')
        except: 
            pass
        
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel("Max FM Strength")
        
        ax.legend()

    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_FM.png", dpi=300)

    # ---------------- c ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        data_by_depth = [df[df["min_depth"]==d]["max_postsynaptic_weight"].values for d in depths]
        ax.boxplot(data_by_depth, positions=depths, widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor="tab:orange", alpha=0.6), showfliers=False,
                   medianprops=dict(color='black', linewidth=1.5),
                   whiskerprops=dict(color="tab:orange", linestyle='--'),
                   capprops=dict(color="tab:orange"))
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel(f"Max Post Strength")
    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_post.png", dpi=300)

    # ---------------- d ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        data_by_depth = [df[df["min_depth"]==d]["max_presynaptic_weight"].values for d in depths]
        ax.boxplot(data_by_depth, positions=depths, widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor="tab:green", alpha=0.6), showfliers=False,
                   medianprops=dict(color='black', linewidth=1.5),
                   whiskerprops=dict(color="tab:green", linestyle='--'),
                   capprops=dict(color="tab:green"))
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel(f"Max Pre Strength")
    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_pre.png", dpi=300)

    plt.show()

plot_Strengths_4_figs(
    left_csv="./output/visual_left_weight_summary.csv",
    right_csv="./output/visual_right_weight_summary.csv",
    save_path_prefix="../results/Figure2/combined"
)

# Supplementary Fig. 6

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
import scienceplots

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.5,
})


def plot_connectivity_stats_vs_depth(csv_path, side, save_path=None):
    df = pd.read_csv(csv_path)
    df = df[df["min_depth"] >= 0]

    depths = np.sort(df["min_depth"].unique())

    metrics = [
        "mean_upstream_weight",
        "count_upstream_nonzero",
        "mean_downstream_weight",
        "count_downstream_nonzero"
    ]

    titles = [
        "Mean Upstream Weight",
        "Upstream Connection Count",
        "Mean Downstream Weight",
        "Downstream Connection Count"
    ]

    colors = [
        "tab:blue",
        "tab:orange",
        "tab:green",
        "tab:red"
    ]

    fig, axes = plt.subplots(1, 4, figsize=(24, 5))

    for ax, metric, color, title in zip(axes, metrics, colors, titles):

        data_by_depth = []

        for d in depths:
            vals = df[df["min_depth"] == d][metric].values
            data_by_depth.append(vals)

        means = np.array([np.mean(v) if len(v) > 0 else np.nan for v in data_by_depth])

        boxprops = dict(facecolor=color, alpha=0.6, linewidth=1.5)
        medianprops = dict(color='black', linewidth=1.5)
        whiskerprops = dict(color=color, linewidth=1.2, linestyle='--')
        capprops = dict(color=color, linewidth=1.2)

        ax.boxplot(
            data_by_depth,
            positions=depths,
            widths=0.6,
            patch_artist=True,
            showfliers=False,
            boxprops=boxprops,
            medianprops=medianprops,
            whiskerprops=whiskerprops,
            capprops=capprops
        )

        for i, vals in enumerate(data_by_depth):
            if len(vals) == 0:
                continue
            x = np.random.normal(depths[i], 0.08, size=len(vals))
            ax.scatter(x, vals, color='black', alpha=0.3, s=20)

        ax.plot(depths, means, marker="o", linewidth=2, color="red", label="Mean")

        ax.set_xlabel("Min Propagation Depth")
        ax.set_title(title)
        ax.set_xlim(min(depths) - 0.5, max(depths) + 0.5)

        if "count" in metric:
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1e'))

        if ax == axes[0]:
            ax.set_ylabel(f"{side}")

        ax.legend(loc="upper right", frameon=False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


plot_connectivity_stats_vs_depth(
    "./output/visual_right_weight_summary.csv",
    side="Right",
    save_path="../results/SupplementaryFigure/right_connectivity_vs_depth_boxplot_scatter.png"
)

plot_connectivity_stats_vs_depth(
    "./output/visual_left_weight_summary.csv",
    side="Left",
    save_path="../results/SupplementaryFigure/left_connectivity_vs_depth_boxplot_scatter.png"
)

# Supplementary Fig. 9

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from matplotlib.colors import LogNorm

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.linewidth": 1.5,
})


def plot_weight_pairwise_density_by_depth(csv_path, save_path=None, side=''):

    df = pd.read_csv(csv_path)

    df = df[df["min_depth"] >= 0]
    df = df[df["min_depth"] <= 5]

    depths = sorted(df["min_depth"].unique())

    weight_pairs = [
        ("max_FM_weight", "max_postsynaptic_weight"),
        ("max_FM_weight", "max_presynaptic_weight"),
        ("max_postsynaptic_weight", "max_presynaptic_weight")
    ]

    label_map = {
        "max_FM_weight": "FM Strength",
        "max_postsynaptic_weight": "Post Strength",
        "max_presynaptic_weight": "Pre Strength"
    }

    titles = [
        "FM vs Post",
        "FM vs Pre",
        "Post vs Pre"
    ]

    n_depth = len(depths)

    # -------- log transform --------
    for col in ["max_FM_weight", "max_postsynaptic_weight", "max_presynaptic_weight"]:
        df[col] = np.log10(df[col] + 1e-12)

    # -------- remove outliers via quantile --------
    x_vals = []
    y_vals = []

    for xw, yw in weight_pairs:
        x_vals.extend(df[xw].values)
        y_vals.extend(df[yw].values)

    xmin, xmax = np.percentile(x_vals, [1, 99])
    ymin, ymax = np.percentile(y_vals, [1, 99])

    # -------- figure --------
    fig, axes = plt.subplots(
        3,
        n_depth,
        figsize=(4*n_depth, 11),
        sharex=True,
        sharey=True
    )

    fig.suptitle(f"{side} Neurons")

    hist_ref = None

    for col, depth in enumerate(depths):

        df_d = df[df["min_depth"] == depth]

        for row, ((xw, yw), title) in enumerate(zip(weight_pairs, titles)):

            ax = axes[row, col]

            x = df_d[xw].values
            y = df_d[yw].values

            h = ax.hist2d(
                x,
                y,
                bins=80,
                cmap="coolwarm",
                norm=LogNorm(),
                range=[[xmin, xmax], [ymin, ymax]]
            )

            if hist_ref is None:
                hist_ref = h

            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)

            ax.set_xlabel(label_map[xw])
            ax.set_ylabel(label_map[yw])

            if col == 0:
                ax.set_title(title)

        axes[0, col].text(
            0.5,
            1.22,
            f"Depth {depth} (n={len(df_d):,})", 
            transform=axes[0, col].transAxes,
            ha="center",
            va="center"
        )

    # ---------- unified colorbar ----------
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])

    cbar = fig.colorbar(
        hist_ref[3],
        cax=cbar_ax
    )

    cbar.set_label("Log Density")

    plt.tight_layout(rect=[0, 0, 0.9, 1])

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


plot_weight_pairwise_density_by_depth(
    "./output/visual_right_weight_summary.csv",
    save_path="Right_weights_pairwise_by_depth.png",
    side='Right'
)

plot_weight_pairwise_density_by_depth(
    "./output/visual_left_weight_summary.csv",
    save_path="Left_weights_pairwise_by_depth.png",
    side='Left'
)

# Fig. 2(e)

In [ ]:
import navis
import flybrains
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from pathlib import Path
from tqdm import tqdm

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 20,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "axes.linewidth": 1.5,
})

# -----------------------------
# paths
# -----------------------------
swc_path = Path("../data/sk_lod1_783_healed")

csv_paths = {
    "right": "./output/visual_right_weight_summary.csv",
    "left": "./output/visual_left_weight_summary.csv"
}

# -----------------------------
# flywire brain mesh
# -----------------------------
brain_mesh = flybrains.FLYWIRE.mesh

depth_colors = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2"
]

# -----------------------------
# scan SWC once
# -----------------------------
print("Scanning SWC directory...")

swc_files = list(swc_path.glob("*.swc"))
id_to_file = {}

for f in swc_files:
    try:
        nid = int(f.stem.split('.')[0])
        id_to_file[nid] = f
    except ValueError:
        continue

print(f"Found {len(id_to_file)} SWC files")


# -----------------------------
# main plotting function
# -----------------------------
def plot_depth_distribution(csv_path, side):

    df = pd.read_csv(csv_path)

    df = df[(df["min_depth"] >= 0) & (df["min_depth"] <= 5)]

    depths = sorted(df["min_depth"].unique())

    print(f"\nProcessing {side} side")

    for i, depth in enumerate(tqdm(depths, desc=f"{side} depths")):

        df_d = df[df["min_depth"] == depth]

        ids = df_d["root_id"].values
        weights = df_d["max_FM_weight"].values

        if len(ids) == 0:
            continue

        # -----------------------------
        # normalize weights
        # -----------------------------
        wmin, wmax = weights.min(), weights.max()

        if wmax - wmin < 1e-12:
            norm_w = np.ones_like(weights)
        else:
            norm_w = (weights - wmin) / (wmax - wmin)

        # -----------------------------
        # find SWC files
        # -----------------------------
        files_to_load = [
            id_to_file[nid] for nid in ids if nid in id_to_file
        ]

        if not files_to_load:
            continue

        skeletons = navis.read_swc(files_to_load, parallel=True)

        id_to_skeleton = {}

        for sk in skeletons:
            try:
                id_to_skeleton[int(sk.id)] = sk
            except:
                continue

        skels = []
        alphas = []

        for j, nid in enumerate(ids):

            if nid in id_to_skeleton:

                skels.append(id_to_skeleton[nid])

                alpha = 0.15 + 0.85 * norm_w[j]
                alphas.append(alpha)

        if not skels:
            continue

        # -----------------------------
        # plotting
        # -----------------------------
        fig, ax = plt.subplots(figsize=(6, 6))

        navis.plot2d(
            brain_mesh,
            view=("x", "-y", "-z"),
            ax=ax,
            color="lightgrey",
            alpha=0.3
        )

        navis.plot2d(
            skels,
            view=("x", "-y", "-z"),
            ax=ax,
            color=depth_colors[i % len(depth_colors)],
            alpha=alphas,
            linewidth=0.3
        )

        ax.set_title(
            f"{side.capitalize()} neurons depth {depth}\n(n = {len(skels)})",
            fontsize=36,
            color="black"
        )

        ax.set_axis_off()

        for spine in ax.spines.values():
            spine.set_visible(False)

        plt.tight_layout()

        plt.savefig(
            f"../results/Figure2/{side}_flywire_depth_{depth}.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.close()


# -----------------------------
# run both sides
# -----------------------------
for side, path in csv_paths.items():
    plot_depth_distribution(path, side)

# Fig. 2(f)

In [ ]:
import csv
import os
from tqdm import tqdm
from collections import Counter

# --- 1. DATA PREPARATION: Neuron ID Retrieval ---
def read_neuron_ids(neuron_type, side=None, filename='../data/column_assignment.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                if row['type'] == neuron_type and (side is None or row['hemisphere'].lower() == side.lower()):
                    try:
                        neuron_ids.append(int(row['root_id']))
                    except ValueError:
                        continue
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return neuron_ids


def get_all_neuron_types(filename='../data/column_assignment.txt'):
    neuron_types = set()
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                neuron_types.add(row['type'])
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return list(neuron_types)


neuron_types = get_all_neuron_types()
sides = ['left', 'right']

tasks = {}   # {(type, side): neuron_ids}

for neuron_type in neuron_types:
    for side in sides:
        neuron_ids = read_neuron_ids(neuron_type=neuron_type, side=side)

        tasks[(neuron_type, side)] = neuron_ids

print("\n========== Task summary ==========")
print(f"Total tasks (type × side): {len(tasks)}")

sizes = [len(v) for v in tasks.values()]
print(f"Neuron count per task: min={min(sizes)}, max={max(sizes)}")

empty = sum(1 for v in sizes if v == 0)
print(f"Empty tasks: {empty}")
print("=================================\n")

import matplotlib.pyplot as plt
import scienceplots
import umap
import matplotlib.colors as mcolors
import numpy as np
from sklearn.metrics import silhouette_score

# --- 2. STYLE CONFIGURATION: Publication Standards ---
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 28,
    "axes.labelsize": 28,
    "axes.titlesize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "axes.linewidth": 1.5,
})

# --- 3. CORE PROCESSING: Feature Extraction & Standardization ---
def plot_neuron_umap_from_profiles(
    tasks: dict,
    title: str = '',
    crop_size: int = 21,
    n_neighbors: int = 100,
    min_dist: float = 0.7,
    scatter_size: float = 10,
    random_state: int = 42,
    save_name: str = "neuron_umap",
    side_list: list = ['left', 'right', 'both']  
):

    layers_list = ['l1', 'l2', 'l3']
    half = crop_size // 2
    save_dir = '../results/Figure2'
    os.makedirs(save_dir, exist_ok=True)

    mask = np.load('./output/combined_eye_mask_41x82.npz')['mask'].astype(bool)
    H, W = mask.shape
    yy, xx = np.where(mask)

    # -------------------------- Load matrices --------------------------
    matrices = {}
    id_maps = {}
    for layer in layers_list:
        matrices[layer] = {}
        id_maps[layer] = {}
        for s in ['left', 'right']:
            path = f'./output/VFM/{layer}_{s}.npz'
            if os.path.exists(path):
                d = np.load(path)
                matrices[layer][s] = {
                    'target_ids': d['target_ids'],
                    'w': (d['exc'] + d['inh']).astype(np.float32)
                }
                id_maps[layer][s] = {nid: i for i, nid in enumerate(d['target_ids'])}
            else:
                matrices[layer][s] = None
                id_maps[layer][s] = {}

    # -------------------------- Collect features --------------------------
    features_dict = {side: {layer: [] for layer in layers_list + ['all']} for side in ['left', 'right', 'both']}
    labels_dict = {side: {layer: [] for layer in layers_list + ['all']} for side in ['left', 'right', 'both']}
    visited = set()

    for (neuron_type, side), neuron_ids in tqdm(tasks.items(), desc='Extracting features'):
        for nid in neuron_ids:
            if nid in visited:
                continue
            visited.add(nid)

            full_layers = []
            max_val = 0.0
            cx = cy = None

            for layer in layers_list:
                full = np.zeros((H, W), dtype=np.float32)
                for s in ['left', 'right']:
                    entry = matrices[layer][s]
                    if entry is None:
                        continue
                    idx = id_maps[layer][s].get(nid)
                    if idx is None:
                        continue
                    w = entry['w'][idx]
                    if s == 'left':
                        full[:, :W//2] = w
                    else:
                        full[:, W//2:] = np.fliplr(w)

                full_layers.append(full)
                flat = full[mask]
                if flat.size == 0:
                    continue
                max_idx = np.argmax(np.abs(flat))
                if np.abs(flat[max_idx]) > max_val:
                    max_val = np.abs(flat[max_idx])
                    cy = yy[max_idx]
                    cx = xx[max_idx]

            if max_val < 1e-12:
                continue

            xs = xx - cx + half
            ys = yy - cy + half
            inside = (xs >= 0) & (xs < crop_size) & (ys >= 0) & (ys < crop_size)

            layer_feats = []
            for full in full_layers:
                patch = np.zeros((crop_size, crop_size), dtype=np.float32)
                patch[ys[inside], xs[inside]] = full[mask][inside]
                layer_feats.append(patch.flatten())

            feat_concat = np.concatenate(layer_feats)

            for i, layer in enumerate(layers_list):
                features_dict[side][layer].append(layer_feats[i])
                labels_dict[side][layer].append(neuron_type)
                features_dict['both'][layer].append(layer_feats[i])
                labels_dict['both'][layer].append(neuron_type)

            features_dict[side]['all'].append(feat_concat)
            labels_dict[side]['all'].append(neuron_type)
            features_dict['both']['all'].append(feat_concat)
            labels_dict['both']['all'].append(neuron_type)

    # --- 4. CLUSTERING & VISUALIZATION ---
    # -------------------------- UMAP + Silhouette Score --------------------------
    embedding_dict = {}
    silhouette_dict = {}
    for side in side_list:
        embedding_dict[side] = {}
        silhouette_dict[side] = {}
        for layer in layers_list + ['all']:
            feats = np.asarray(features_dict[side][layer])
            lbls = np.asarray(labels_dict[side][layer])
            if feats.shape[0] == 0:
                embedding_dict[side][layer] = None
                silhouette_dict[side][layer] = None
                continue

            reducer = umap.UMAP(
                n_neighbors=n_neighbors,
                min_dist=min_dist,
                n_components=2,
                random_state=random_state
            )
            embedding_dict[side][layer] = reducer.fit_transform(feats)

            try:
                score = silhouette_score(
                    feats,
                    lbls,
                    sample_size=min(2000, len(lbls)),
                    random_state=0
                )
            except Exception:
                score = None
            silhouette_dict[side][layer] = score
            if score is not None:
                print(f"Silhouette Score - side: {side}, layer: {layer} -> {score:.3f}")

    # -------------------------- Nature 40-color palette --------------------------
    nature_40 = [
        "#4C72B0","#55A868","#C44E52","#8172B3","#CCB974","#64B5CD",
        "#8C8C8C","#E17C05","#5DA5DA","#B276B2","#F17CB0","#60BD68",
        "#F15854","#4D4D4D","#B2912F","#499894","#E15759","#79706E",
        "#86BCB6","#FF9DA7","#9C755F","#3182bd","#31a354","#74c476","#a1d99b",
        "#756bb1","#9e9ac8","#bcbddc","#636363"
    ]

    # -------------------------- Plot --------------------------
    valid_sides = [s for s in ['left', 'right', 'both'] if s in side_list]
    n_rows = len(valid_sides)
    fig, axes = plt.subplots(n_rows, 4, figsize=(18, 5*n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, 0)

    layer_names = ['L1', 'L2', 'L3', 'All']
    scatter_handles = []

    for row_idx, side in enumerate(valid_sides):
        for col_idx, layer_name in enumerate(layer_names):
            ax = axes[row_idx, col_idx]
            key = layer_name.lower() if layer_name != 'All' else 'all'
            emb = embedding_dict[side][key]
            lbls = labels_dict[side][key]
            score = silhouette_dict[side][key]

            if emb is None or len(lbls) == 0:
                ax.axis('off')
                continue

            unique_labels = sorted(set(lbls))
            num_types = len(unique_labels)

            if num_types <= len(nature_40):
                colors_list = nature_40[:num_types]
            else:
                n_colors = len(nature_40)
                colors_list = []
                for i in range(num_types):
                    base_color = nature_40[i % n_colors]
                    rgb = np.array(mcolors.to_rgb(base_color))
                    factor = 0.9 ** (i // 40)
                    colors_list.append(rgb * factor)

            lbls_np = np.array(lbls)
            for i, t in enumerate(unique_labels):
                idx = np.where(lbls_np == t)[0]
                sc = ax.scatter(
                    emb[idx, 0],
                    emb[idx, 1],
                    s=scatter_size,
                    alpha=0.3,
                    color=colors_list[i],
                    edgecolors='none',
                    linewidths=0,
                    label=t if row_idx == n_rows-1 and layer_name == 'All' else None
                )
                if row_idx == n_rows-1 and layer_name == 'All':
                    scatter_handles.append(sc)

            if score is not None:
                ax.set_title(f"{layer_name}\nSilhouette: {score:.3f}")
            else:
                ax.set_title(f"{layer_name}\nSilhouette: N/A")

            ax.set_xticks([])
            ax.set_yticks([])
            ax.axis('off')

    row_labels = {'left':'Left','right':'Right','both':'Both'}
    for i, side in enumerate(valid_sides):
        y = 1 - (i + 0.5)/n_rows
        fig.text(0.02, y, row_labels[side]+title, rotation=90, va='center', ha='center')

    from matplotlib.lines import Line2D
    labels = [s.get_label() for s in scatter_handles]
    colors = [s.get_facecolor()[0] for s in scatter_handles]
    proxy_handles = [Line2D([0], [0], marker='o', color='none', 
                            markerfacecolor=c, alpha=1, markersize=10)
                    for c in colors]
    n_per_col = 60
    N = len(labels)
    import math
    ncol = math.ceil(N / n_per_col)

    new_labels = []
    new_handles = []
    for r in range(n_per_col):
        for c in range(ncol):
            idx = c * n_per_col + r
            if idx < N:
                new_labels.append(labels[idx])
                new_handles.append(proxy_handles[idx])
    
    fig.legend(
        handles=new_handles,
        labels=new_labels,
        loc='center left',
        bbox_to_anchor=(1, 0.5),
        bbox_transform=fig.transFigure,
        frameon=False,
        fontsize=20,
        markerscale=2,
        ncol=ncol
    )

    plt.tight_layout(rect=[0.05, 0.05, 0.93, 0.95])
    plt.savefig(os.path.join(save_dir, f'{save_name}.pdf'), bbox_inches='tight')
    plt.show()

    return embedding_dict, silhouette_dict

# --- 5. EXECUTION ---
# -------------------------------
# UMAP
# -------------------------------
plot_neuron_umap_from_profiles(
    tasks=tasks,
    crop_size=5,
    save_name="column_umap"
)

# Supplementary Fig. 4

In [ ]:
import csv
import os
from tqdm import tqdm
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"], 
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.linewidth": 1.5,
})

def read_neuron_ids(neuron_type, side=None, filename='../data/column_assignment.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                if row['type'] == neuron_type and (side is None or row['hemisphere'].lower() == side.lower()):
                    try:
                        neuron_ids.append(int(row['root_id']))
                    except ValueError:
                        continue
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return neuron_ids


def get_all_neuron_types(filename='../data/column_assignment.txt'):
    neuron_types = set()
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                neuron_types.add(row['type'])
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return list(neuron_types)

neuron_types = get_all_neuron_types()
sides = ['left', 'right']

tasks = {}   # {(type, side): neuron_ids}

for neuron_type in neuron_types:
    for side in sides:
        neuron_ids = read_neuron_ids(neuron_type=neuron_type, side=side)

        tasks[(neuron_type, side)] = neuron_ids


print("\n========== Task summary ==========")
print(f"Total tasks (type × side): {len(tasks)}")

sizes = [len(v) for v in tasks.values()]
print(f"Neuron count per task: min={min(sizes)}, max={max(sizes)}")

empty = sum(1 for v in sizes if v == 0)
print(f"Empty tasks: {empty}")
print("=================================\n")

def plot_type_mean_correlation(
    tasks: dict,
    crop_size: int = 21,
    save_name: str = "type_mean_correlation_advanced"
):

    layers_list = ['l1', 'l2', 'l3']
    layer_names = ['L1', 'L2', 'L3', 'All']
    half = crop_size // 2

    save_dir = '../results/SupplementaryFigure/'
    os.makedirs(save_dir, exist_ok=True)

    mask = np.load('./output/combined_eye_mask_41x82.npz')['mask'].astype(bool)
    H, W = mask.shape
    yy, xx = np.where(mask)

    matrices = {}
    for layer in layers_list:
        matrices[layer] = {}
        for s in ['left', 'right']:
            path = f'./output/VFM/{layer}_{s}.npz'
            if os.path.exists(path):
                d = np.load(path)
                matrices[layer][s] = {
                    'target_ids': d['target_ids'],
                    'w': (d['exc'] + d['inh']).astype(np.float32)
                }
            else:
                matrices[layer][s] = None

    features_dict = {side: {layer: {} for layer in layers_list + ['all']}
                    for side in ['left', 'right', 'both']}

    visited = set()
    for (neuron_type, side), neuron_ids in tqdm(tasks.items(), desc="Extracting"):
        for nid in neuron_ids:
            if nid in visited:
                continue
            visited.add(nid)

            full_layers = []
            max_val = 0.0
            cx = cy = None

            for layer in layers_list:
                full = np.zeros((H, W), dtype=np.float32)
                for s in ['left', 'right']:
                    entry = matrices[layer][s]
                    if entry is None:
                        continue
                    ids = entry['target_ids']
                    idx = np.where(ids == nid)[0]
                    if idx.size == 0:
                        continue
                    w = entry['w'][idx[0]]
                    if s == 'left':
                        full[:, :W//2] = w
                    else:
                        full[:, W//2:] = np.fliplr(w)
                full_layers.append(full)

                abs_valid = np.abs(full)[mask]
                if abs_valid.size == 0:
                    continue
                if abs_valid.max() > max_val:
                    max_val = abs_valid.max()
                    idx_max = np.argmax(abs_valid)
                    cy = yy[idx_max]
                    cx = xx[idx_max]

            if max_val < 1e-12:
                continue

            xs = xx - cx + half
            ys = yy - cy + half
            inside = (xs >= 0) & (xs < crop_size) & (ys >= 0) & (ys < crop_size)

            layer_feats = []
            for full in full_layers:
                patch = np.zeros((crop_size, crop_size), dtype=np.float32)
                patch[ys[inside], xs[inside]] = full[mask][inside]
                layer_feats.append(patch.flatten())

            feat_concat = np.concatenate(layer_feats)

            for i, layer in enumerate(layers_list):
                for s in [side, 'both']:
                    features_dict[s][layer].setdefault(neuron_type, []).append(layer_feats[i])
            for s in [side, 'both']:
                features_dict[s]['all'].setdefault(neuron_type, []).append(feat_concat)


        
    # --------------------------
    # Compute correlations
    # --------------------------
    corr_dict = {side: {layer: {} for layer in layers_list + ['all']}
                 for side in ['left', 'right', 'both']}

    for side in ['left', 'right', 'both']:
        for layer in layers_list + ['all']:
            for t, feats in features_dict[side][layer].items():
                feats = np.array(feats)
                if np.all(feats == 0):
                    corr_dict[side][layer][t] = [0] * len(feats)
                    continue
                mean_pattern = np.mean(feats, axis=0)
                corrs = []
                for v in feats:
                    if np.std(v) == 0 or np.std(mean_pattern) == 0:
                        corrs.append(0) 
                    else:
                        corrs.append(np.corrcoef(v, mean_pattern)[0, 1])
                corr_dict[side][layer][t] = corrs

    # --------------------------
    # Plot
    # --------------------------
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    palette = ['#4C72B0', '#55A868', '#C44E52', '#8172B3']
    width = 0.18

    for row_idx, side in enumerate(['left', 'right', 'both']):
        ax = axes[row_idx]
        types = sorted(corr_dict[side]['all'].keys())
        x_base = np.arange(len(types))

        for i, layer in enumerate(['l1', 'l2', 'l3', 'all']):
            offsets = x_base + (i - 1.5) * width
            data = [corr_dict[side][layer].get(t, [0]) for t in types]

            # violin
            parts = ax.violinplot(
                data,
                positions=offsets,
                widths=width,
                showmeans=False,
                showmedians=False,
                showextrema=False
            )
            for pc in parts['bodies']:
                pc.set_facecolor(palette[i])
                pc.set_alpha(0.3)

            # box
            box = ax.boxplot(
                data,
                positions=offsets,
                widths=width * 0.6,
                patch_artist=True,
                showfliers=False
            )
            for patch in box['boxes']:
                patch.set_facecolor("white")
                patch.set_edgecolor(palette[i])
                patch.set_linewidth(1.2)

            # jitter scatter
            for j, vals in enumerate(data):
                if len(vals) == 0:
                    continue
                x_jitter = np.random.normal(offsets[j], width * 0.08, size=len(vals))
                ax.scatter(x_jitter, vals,
                           s=10,
                           alpha=0.05,
                           color=palette[i],
                           edgecolors='none')

        ax.set_ylim(-1, 1)
        ax.set_ylabel("Pearson r")
        ax.set_title(side.capitalize())

    for i, layer in enumerate(['L1', 'L2', 'L3', 'All']):
        axes[0].scatter([], [], color=palette[i], label=layer, alpha=0.6, s=50)

    axes[0].legend(
        title="Layer",
        fontsize=16,
        title_fontsize=18,
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0,
        markerscale=3
    )

    axes[-1].set_xticks(np.arange(len(types)))
    axes[-1].set_xticklabels(types, rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{save_name}.pdf"), bbox_inches='tight')
    plt.show()

plot_type_mean_correlation(
    tasks=tasks,
    crop_size=5,
    save_name="heatmap"
)

# Fig. 2(i)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

plt.style.use(['science','nature','no-latex'])
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 28,
    "axes.labelsize": 28,
    "axes.titlesize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "axes.linewidth": 1.5,
})

depths = [0,1,2,3,4]

color_map = {
    "left":"#1E3A8A",   
    "right":"#8B0000"  
}

def prepare_merged(csv_path, column_file):
    column_df = pd.read_csv(column_file, sep=",")
    df = pd.read_csv(csv_path)
    merged = column_df.merge(
        df[['root_id','min_depth']],
        on='root_id',
        how='inner'
    )
    merged = merged[merged['min_depth'].between(0,4)]
    return merged

def compute_stats(merged, hemi_filter):
    stats = (
        merged[merged['hemisphere']==hemi_filter]
        .groupby(['type','hemisphere','min_depth'])
        .size()
        .reset_index(name="count")
    )
    totals = (
        merged[merged['hemisphere']==hemi_filter]
        .groupby(['type','hemisphere'])
        .size()
        .reset_index(name="total")
    )
    stats = stats.merge(totals,on=['type','hemisphere'])
    stats["ratio"] = stats["count"] / stats["total"]
    return stats

def plot_mirror_bubble(left_csv, right_csv, column_file, save_path=None):

    left_merged = prepare_merged(left_csv,column_file)
    right_merged = prepare_merged(right_csv,column_file)

    left_stats = compute_stats(left_merged,'left')
    right_stats = compute_stats(right_merged,'right')

    types = np.sort(np.unique(np.concatenate([left_merged['type'].unique(), right_merged['type'].unique()])))
    types = types[::-1]

    y_pos = np.arange(len(types))
    fig, ax = plt.subplots(figsize=(12,len(types)*0.5))

    max_ratio = max(left_stats["ratio"].max(), right_stats["ratio"].max())
    size_scale = 800 / max_ratio  

    for i,t in enumerate(types):
        y = y_pos[i]
        lsub = left_stats[left_stats["type"]==t]
        for _,row in lsub.iterrows():
            depth = row["min_depth"]
            ratio = row["ratio"]
            hemi = row["hemisphere"]
            x = -depth
            ax.scatter(
                x,y,
                s=ratio*size_scale,
                color=color_map[hemi],
                alpha=0.3 + 0.6*ratio,
                edgecolor="black",
                linewidth=0.5
            )

    for i,t in enumerate(types):
        y = y_pos[i]
        rsub = right_stats[right_stats["type"]==t]
        for _,row in rsub.iterrows():
            depth = row["min_depth"]
            ratio = row["ratio"]
            hemi = row["hemisphere"]
            x = depth
            ax.scatter(
                x,y,
                s=ratio*size_scale,
                color=color_map[hemi],
                alpha=0.3 + 0.6*ratio,
                edgecolor="black",
                linewidth=0.5
            )

    ax.axvline(0,color="black",linewidth=1)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(types)

    xticks = [-4,-3,-2,-1,0,1,2,3,4]
    xticklabels = [4,3,2,1,0,1,2,3,4]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)
    ax.set_xlabel("Depth (Left ← → Right)")

    from matplotlib.lines import Line2D
    legend_elements = [
    ]

    for s in [0.1,0.3,0.6]:
        legend_elements.append(Line2D([0],[0],marker='o',color='w',label=f'{int(s*100)}% proportion',markerfacecolor='gray',markersize=np.sqrt(s*size_scale),alpha=0.6))

    ax.legend(handles=legend_elements,loc='center left', title='Proportion of Neuron Counts', bbox_to_anchor=(1,0.5),frameon=True,fontsize=22,title_fontsize=20)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path,dpi=600,bbox_inches="tight")
    plt.show()

plot_mirror_bubble(
    "./output/visual_left_weight_summary.csv",
    "./output/visual_right_weight_summary.csv",
    "../data/column_assignment.txt",
    save_path="../results/Figure2/mirror_depth_bubble_same_side.png"
)